In [2]:
%%writefile user_data.csv
user_id,series,season,timestamp,genre,duration_mins
521,”Mirzapur”,3,2024-07-30 15:00:00,action,300
672,”Panchayat”,3,2024-07-30 15:00:00,comedy,200
197,”Family Man”,2,2024-07-30 15:00:00,action,500
521,”Mirzapur”,2,2024-07-29 15:00:00,action,280
211,”Queens Gambit”,1,2024-07-30 15:00:00,drama,170
521,”Mirzapur”,1,2024-07-28 15:00:00,action,230
844,”Westworld”,3,2024-07-30 15:00:00,sci-fi,310
672,”Panchayat”,3,2024-07-29 15:00:00,comedy,210
256,”Homecoming”,2,2024-07-30 15:00:00,thriller,310
489,”Outer Range”,1,2024-07-30 15:00:00,sci-fi,340
200,”Black Mirror”,2,2024-07-30 15:00:00,sci-fi,140
256,”Outer Range”,2,2024-07-30 15:00:00,thriller,250
489,”Outer Range”,2,2024-07-28 15:00:00,sci-fi,170
200,”Black Mirror”,3,2024-07-29 15:00:00,sci-fi,190
672,”Panchayat”,2,2024-07-28 15:00:00,comedy,160
672,”Outer Range”,1,2024-07-25 15:00:00,sci-fi,250
200,”Black Mirror”,4,2024-07-28 15:00:00,sci-fi,200
844,”Westworld”,2,2024-07-29 15:00:00,sci-fi,300
672,”Black Mirror”,5,2024-07-28 15:00:00,sci-fi,150
672,”Panchayat”,1,2024-07-27 15:00:00,comedy,190

Writing user_data.csv


In [12]:
#Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [4]:
#Create SparkSession for app (Streaming Analysis)
spark= SparkSession.builder.appName("Streaming Analysis").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/07 04:13:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [10]:
#Create Dataframe series_df
df = spark.read.csv("user_data.csv", header=True, inferSchema=True)
df.show()

df.printSchema()

+-------+---------------+------+-------------------+--------+-------------+
|user_id|         series|season|          timestamp|   genre|duration_mins|
+-------+---------------+------+-------------------+--------+-------------+
|    521|     ”Mirzapur”|     3|2024-07-30 15:00:00|  action|          300|
|    672|    ”Panchayat”|     3|2024-07-30 15:00:00|  comedy|          200|
|    197|   ”Family Man”|     2|2024-07-30 15:00:00|  action|          500|
|    521|     ”Mirzapur”|     2|2024-07-29 15:00:00|  action|          280|
|    211|”Queens Gambit”|     1|2024-07-30 15:00:00|   drama|          170|
|    521|     ”Mirzapur”|     1|2024-07-28 15:00:00|  action|          230|
|    844|    ”Westworld”|     3|2024-07-30 15:00:00|  sci-fi|          310|
|    672|    ”Panchayat”|     3|2024-07-29 15:00:00|  comedy|          210|
|    256|   ”Homecoming”|     2|2024-07-30 15:00:00|thriller|          310|
|    489|  ”Outer Range”|     1|2024-07-30 15:00:00|  sci-fi|          340|
|    200| ”B

In [14]:
#Find the user with maximum watchtime
df.orderBy(F.col("duration_mins").desc()).select("user_id").limit(1).show()

+-------+
|user_id|
+-------+
|    197|
+-------+



In [18]:
#Calculate overall total Watchtime
df.select(F.sum("duration_mins")).show()

+------------------+
|sum(duration_mins)|
+------------------+
|              4850|
+------------------+



In [19]:
#Find most popular shows (based on watchtime)
df.orderBy(F.col("duration_mins").desc()).select("series","season").limit(1).show()

+------------+------+
|      series|season|
+------------+------+
|”Family Man”|     2|
+------------+------+



In [25]:
#Find most popular shows (based on user popularity)
max1=df.groupby("series","season").count().agg(F.max("count")).first()[0]
df.groupby("series","season").count().orderBy(F.col("count").desc()).filter(F.col("count")==max1).select("series","season").show()

+-------------+------+
|       series|season|
+-------------+------+
|  ”Panchayat”|     3|
|”Outer Range”|     2|
|”Outer Range”|     1|
+-------------+------+



In [29]:
#Find the most popular genre
df.groupby("genre").count().orderBy(F.col("count").desc()).limit(1).show()

+------+-----+
| genre|count|
+------+-----+
|sci-fi|    9|
+------+-----+



In [30]:
#Calculate total watchtime per user
df.groupby("user_id").agg(F.sum("duration_mins")).show()

+-------+------------------+
|user_id|sum(duration_mins)|
+-------+------------------+
|    211|               170|
|    844|               610|
|    489|               510|
|    197|               500|
|    672|              1160|
|    256|               560|
|    200|               530|
|    521|               810|
+-------+------------------+



In [32]:
#Find most popular genre (based on engagement count)
df.groupby("user_id","genre").count().orderBy(F.col("count").desc()).limit(1).show()

+-------+------+-----+
|user_id| genre|count|
+-------+------+-----+
|    672|comedy|    4|
+-------+------+-----+



In [34]:
#Find average watchtime per genre
df.groupby("genre").agg(F.round(F.avg("duration_mins"),1)).show()

+--------+----------------------------+
|   genre|round(avg(duration_mins), 1)|
+--------+----------------------------+
|  action|                       327.5|
|   drama|                       170.0|
|thriller|                       280.0|
|  sci-fi|                       227.8|
|  comedy|                       190.0|
+--------+----------------------------+



In [42]:
#Find peak traffic days

#(Output 1 = Full Date)
df.withColumn("Date", F.to_date("timestamp")).show()

#(Output 2 = Only Day)
df.withColumn("Day", F.dayname(F.to_date("timestamp"))).show()


+-------+---------------+------+-------------------+--------+-------------+----------+
|user_id|         series|season|          timestamp|   genre|duration_mins|      Date|
+-------+---------------+------+-------------------+--------+-------------+----------+
|    521|     ”Mirzapur”|     3|2024-07-30 15:00:00|  action|          300|2024-07-30|
|    672|    ”Panchayat”|     3|2024-07-30 15:00:00|  comedy|          200|2024-07-30|
|    197|   ”Family Man”|     2|2024-07-30 15:00:00|  action|          500|2024-07-30|
|    521|     ”Mirzapur”|     2|2024-07-29 15:00:00|  action|          280|2024-07-29|
|    211|”Queens Gambit”|     1|2024-07-30 15:00:00|   drama|          170|2024-07-30|
|    521|     ”Mirzapur”|     1|2024-07-28 15:00:00|  action|          230|2024-07-28|
|    844|    ”Westworld”|     3|2024-07-30 15:00:00|  sci-fi|          310|2024-07-30|
|    672|    ”Panchayat”|     3|2024-07-29 15:00:00|  comedy|          210|2024-07-29|
|    256|   ”Homecoming”|     2|2024-07-30 

In [ ]:
#Find the user with most diverse show preference
df.groupby("user_id")

In [ ]:
#Find the binge-watchers


In [ ]:
#Find the user with longest watching streak


In [ ]:
#Total Seasons available


In [ ]:
#Fetch a list of all series
